# 13 — Tool Discovery：动态加载自定义工具

前面的章节里，每加一个新工具都要手动修改主代码，把工具类 `import` 进来，再调用 `registry.register()`。

这章换一种方式：**工具自动发现**。把工具文件放到指定目录，Agent 启动时自动扫描、加载、注册。不需要动任何框架代码。

```
.ai_agent/
└── tools/
    ├── run_pytest.py      ← 放进去
    ├── git_log.py         ← 放进去
    └── slack_notify.py    ← 放进去
```

Agent 启动：三个工具全部自动可用。

---

这章涉及的技术：
- `importlib`：Python 动态模块加载
- `dir()` + `inspect`：从模块中提取工具类
- 错误隔离：一个文件加载失败不影响其他工具

## 1. 为什么需要插件机制

### 静态注册的问题

目前的工具注册流程：

```python
# agent.py  ← 每次加工具都要改这个文件
from src.file_tools import ReadFileTool, WriteFileTool
from src.network_tools import WebSearchTool
from my_custom_tools import RunPytestTool  # 新工具

registry = ToolRegistry()
registry.register(ReadFileTool(cwd))
registry.register(WriteFileTool(cwd))
registry.register(WebSearchTool())
registry.register(RunPytestTool())  # 新工具
```

问题：
1. 框架代码和业务工具耦合
2. 多人协作时容易冲突
3. 运行时不能热加载

### 插件机制的流程

```
用户写工具文件
      │
      ▼
  .ai_agent/tools/run_pytest.py
      │
      ▼
  Agent 启动时
      │
      ├── 扫描目录：glob("*.py")
      │
      ├── 对每个文件：importlib 动态加载
      │
      ├── 从模块里找继承自 Tool 的类
      │
      └── 实例化并注册到 ToolRegistry
```

框架代码完全不变。工具的增删只在 `.ai_agent/tools/` 目录里操作。

In [ ]:
import sys
import importlib.util
import inspect
import logging
import traceback
from pathlib import Path
from typing import Any

sys.path.insert(0, "..")

from src.tool_framework import Tool, ToolKind, ToolResult, ToolRegistry

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
logger = logging.getLogger(__name__)

print("导入完成")

## 2. 动态加载模块：importlib

Python 的 `import` 语句只能加载已知路径下的模块。要在运行时加载任意路径的 `.py` 文件，需要用 `importlib.util`。

三步流程：

```
文件路径
   │
   ▼  importlib.util.spec_from_file_location()
模块规范 (ModuleSpec)
   │  ── 记录模块名、文件位置、加载器
   │
   ▼  importlib.util.module_from_spec()
模块对象 (ModuleType)
   │  ── 空壳：有 __name__、__file__ 等属性，但代码还没执行
   │
   ▼  spec.loader.exec_module()
已加载模块
   │  ── 代码已执行，所有定义的类/函数/变量都在 module 里了
   ▼
  可以用 getattr(module, 'ClassName') 取出里面的东西
```

In [ ]:
def load_module_from_file(file_path: Path):
    """
    从指定 .py 文件动态加载模块。
    
    Args:
        file_path: .py 文件的绝对或相对路径
    Returns:
        已加载的模块对象
    Raises:
        ImportError / SyntaxError / 其他异常：由调用方处理
    """
    # 1. 创建模块规范：告诉 Python 这个模块叫什么、在哪里
    spec = importlib.util.spec_from_file_location(
        name=file_path.stem,        # 模块名 = 文件名（不含 .py）
        location=str(file_path),    # 文件路径
    )
    
    if spec is None or spec.loader is None:
        raise ImportError(f"无法为 {file_path} 创建模块规范")
    
    # 2. 从规范创建空模块对象（代码还没执行）
    module = importlib.util.module_from_spec(spec)
    
    # 3. 执行模块代码（这一步之后，模块里定义的所有东西才可用）
    spec.loader.exec_module(module)
    
    return module


print("load_module_from_file 定义完成")
print()
print("关键概念:")
print("  spec_from_file_location  →  模块规范（描述这个模块是谁、在哪）")
print("  module_from_spec         →  创建空模块对象")
print("  exec_module              →  执行代码，类/函数才真正存在于模块中")

## 3. 从模块中提取工具类

模块加载完之后，用 `dir()` 列出所有属性，过滤出继承自 `Tool` 的类。

三个过滤条件缺一不可：

| 条件 | 原因 |
|------|------|
| `isinstance(obj, type)` | 确保是类，不是函数或变量 |
| `issubclass(obj, Tool)` | 确保是工具类 |
| `obj is not Tool` | 排除 Tool 基类本身（可能被 import 进来） |

In [ ]:
def find_tools_in_module(module) -> list[type]:
    """
    扫描模块，返回所有继承自 Tool 的类（不含 Tool 基类本身）。
    
    Args:
        module: 已加载的模块对象
    Returns:
        工具类列表（是类本身，不是实例）
    """
    tools = []
    for name in dir(module):            # dir() 返回模块里所有属性名
        obj = getattr(module, name)
        if (
            isinstance(obj, type)       # 是个类（不是函数、变量、模块）
            and issubclass(obj, Tool)   # 继承自 Tool
            and obj is not Tool         # 不是 Tool 基类本身
        ):
            tools.append(obj)
    return tools


# 快速验证：用一个内存中的模块测试
class FakeTool(Tool):
    """临时测试用，不实现任何方法"""
    @property
    def name(self): return "fake"
    @property  
    def description(self): return "测试"
    @property
    def kind(self): return ToolKind.READ
    def schema(self): return {}
    async def execute(self, params): return ToolResult.ok("ok")
    def validate(self, params): return []


# 构造一个假模块对象来测试
import types
fake_module = types.ModuleType("fake_module")
fake_module.FakeTool = FakeTool
fake_module.Tool = Tool           # 模块里也 import 了基类
fake_module.some_variable = 42    # 混入一个变量
fake_module.some_function = lambda: None  # 混入一个函数

found = find_tools_in_module(fake_module)
print("在 fake_module 中找到的工具类:")
for cls in found:
    print(f"  {cls.__name__}")
print()
print(f"正确排除了: Tool 基类、变量 (42)、函数 (lambda)")

## 4. 写一个自定义工具文件

先写一个真实的自定义工具，后面用它演示自动发现。

`RunPytestTool` 运行 pytest 测试，返回测试结果。这是 Coding Agent 的常见需求——写完代码后自动验证。

In [ ]:
import os

# 创建插件目录
plugins_dir = Path(".") / ".ai_agent" / "tools"
plugins_dir.mkdir(parents=True, exist_ok=True)

print(f"插件目录: {plugins_dir.resolve()}")

In [ ]:
# 写入 RunPytestTool 工具文件
# 这就是用户实际要写的文件——框架代码完全不需要改动

run_pytest_code = '''"""
自定义工具：运行 pytest 测试。

放到 .ai_agent/tools/ 目录后，Agent 启动时自动发现并注册。
"""
import asyncio
import sys
from pathlib import Path

# 工具文件需要能 import 到框架的基类
# 实际使用时，agent 的 sys.path 里已经包含了 src/，这里显式添加一下
sys.path.insert(0, str(Path(__file__).parent.parent.parent))

from src.tool_framework import Tool, ToolKind, ToolResult


class RunPytestTool(Tool):
    """运行 pytest 测试，返回详细的测试结果。"""

    @property
    def name(self) -> str:
        return "run_pytest"

    @property
    def description(self) -> str:
        return (
            "运行 pytest 测试。返回每条测试的通过/失败状态和失败详情。"
            "适用于验证代码修改是否破坏了现有测试。"
        )

    @property
    def kind(self) -> ToolKind:
        return ToolKind.SHELL  # 执行子进程，属于 SHELL 类型

    def schema(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {
                            "type": "string",
                            "description": "测试路径，可以是目录或具体的测试文件。默认为当前目录。",
                        },
                        "keyword": {
                            "type": "string",
                            "description": "只运行名称匹配这个关键词的测试（pytest -k 参数）。",
                        },
                        "verbose": {
                            "type": "boolean",
                            "description": "是否输出详细信息，默认 True。",
                        },
                    },
                    "required": [],
                },
            },
        }

    def validate(self, params: dict) -> list[str]:
        errors = []
        if "path" in params:
            p = Path(params["path"])
            if not p.exists():
                errors.append(f"路径不存在: {params[\"path\"]")
        return errors

    async def execute(self, params: dict) -> ToolResult:
        # 构造 pytest 命令
        cmd = [sys.executable, "-m", "pytest"]

        test_path = params.get("path", ".")
        cmd.append(test_path)

        if params.get("verbose", True):
            cmd.append("-v")

        if "keyword" in params:
            cmd.extend(["-k", params["keyword"]])

        # 异步执行子进程
        proc = await asyncio.create_subprocess_exec(
            *cmd,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE,
        )
        stdout, stderr = await proc.communicate()

        output = stdout.decode("utf-8", errors="replace")
        err_output = stderr.decode("utf-8", errors="replace")
        full_output = output + ("\n" + err_output if err_output.strip() else "")

        if proc.returncode == 0:
            return ToolResult.ok(full_output, exit_code=0, path=test_path)
        else:
            # returncode=1 表示有测试失败，2 表示中断，5 表示没有找到测试
            return ToolResult.error(
                full_output,
                exit_code=proc.returncode,
                path=test_path,
            )
'''

run_pytest_path = plugins_dir / "run_pytest.py"
run_pytest_path.write_text(run_pytest_code.strip())

print(f"已写入: {run_pytest_path}")

## 5. ToolDiscoveryManager

核心类。负责扫描目录、加载模块、提取工具类、注册到 `ToolRegistry`。

```
tools_dir/
  ├── run_pytest.py    ← load → RunPytestTool → register
  ├── git_log.py       ← load → GitLogTool    → register
  └── __init__.py      ← 跳过（不扫描）
```

每个文件独立加载，互不影响。一个文件加载失败，其他文件继续。失败信息收集到 `errors` 列表，供排查用。

In [ ]:
class ToolDiscoveryManager:
    """
    工具自动发现管理器。
    
    扫描 tools_dir 下的所有 .py 文件，找出继承自 Tool 的类，
    实例化后注册到 registry。
    
    一个文件加载失败不影响其他文件（错误隔离）。
    """

    # 跳过这些特殊文件名
    SKIP_FILES = {"__init__.py", "__main__.py"}

    def __init__(self, tools_dir: Path, registry: ToolRegistry):
        """
        Args:
            tools_dir: 存放自定义工具 .py 文件的目录
            registry:  工具注册表，发现的工具会注册到这里
        """
        self.tools_dir = Path(tools_dir)
        self.registry = registry
        self._errors: list[dict] = []   # 加载失败的记录

    # ── 核心方法 ─────────────────────────────────────────────

    def discover(self) -> list[str]:
        """
        扫描 tools_dir，加载所有工具文件。
        
        Returns:
            成功注册的工具名列表
        """
        self._errors = []           # 每次 discover 重置错误列表
        registered: list[str] = []

        if not self.tools_dir.exists():
            logger.info(f"工具目录不存在，跳过扫描: {self.tools_dir}")
            return registered

        # 按文件名排序，保证每次加载顺序一致
        py_files = sorted(self.tools_dir.glob("*.py"))

        for file_path in py_files:
            # 跳过特殊文件
            if file_path.name in self.SKIP_FILES:
                continue

            # 对每个文件独立加载，失败不影响其他文件
            names = self._load_file(file_path)
            registered.extend(names)

        logger.info(
            f"发现完成：扫描 {len(py_files)} 个文件，"
            f"注册 {len(registered)} 个工具，"
            f"失败 {len(self._errors)} 个文件"
        )
        return registered

    def get_discovery_errors(self) -> list[dict]:
        """
        返回上一次 discover() 中加载失败的文件信息。
        
        每条记录格式:
          {"file": Path, "error": str, "traceback": str}
        """
        return list(self._errors)

    # ── 内部方法 ─────────────────────────────────────────────

    def _load_file(self, file_path: Path) -> list[str]:
        """
        加载单个文件，返回成功注册的工具名列表。
        加载失败时把错误记录到 self._errors，返回空列表。
        """
        try:
            # 动态加载模块
            module = load_module_from_file(file_path)

            # 从模块中找出所有工具类
            tool_classes = find_tools_in_module(module)

            if not tool_classes:
                logger.debug(f"{file_path.name}: 未找到 Tool 子类，跳过")
                return []

            # 实例化并注册
            names = []
            for cls in tool_classes:
                instance = cls()             # 不带参数实例化
                self.registry.register(instance)
                names.append(instance.name)
                logger.info(f"已注册工具: {instance.name}  (来自 {file_path.name})")

            return names

        except Exception as e:
            # 记录错误，继续处理下一个文件
            tb = traceback.format_exc()
            self._errors.append({
                "file": file_path,
                "error": f"{type(e).__name__}: {e}",
                "traceback": tb,
            })
            logger.warning(f"加载失败: {file_path.name}  →  {type(e).__name__}: {e}")
            return []


print("ToolDiscoveryManager 定义完成")

## 6. 演示：完整的工具发现流程

把前面所有部分串起来，走一遍完整流程。

In [ ]:
# 准备：新建一个干净的注册表
registry = ToolRegistry()
print(f"初始注册表: {registry}")
print()

# 创建发现管理器
manager = ToolDiscoveryManager(
    tools_dir=Path(".") / ".ai_agent" / "tools",
    registry=registry,
)

# 执行发现
print("开始扫描...")
registered_names = manager.discover()

print()
print(f"成功注册的工具: {registered_names}")
print(f"错误: {manager.get_discovery_errors()}")
print()
print(f"当前注册表: {registry}")
print(f"工具数量: {len(registry)}")

In [ ]:
# 验证工具已注册，可以调用
tool = registry.get("run_pytest")
print(f"获取到的工具: {tool}")
print(f"名称: {tool.name}")
print(f"描述: {tool.description}")
print(f"类型: {tool.kind}")
print(f"是否修改状态: {tool.is_mutating()}")

In [ ]:
# 写一个简单的测试文件，用 run_pytest 实际运行它
test_code = '''
def test_addition():
    assert 1 + 1 == 2

def test_string_upper():
    assert "hello".upper() == "HELLO"

def test_list_length():
    items = [1, 2, 3]
    assert len(items) == 3
'''

test_file = Path(".") / "_test_demo.py"
test_file.write_text(test_code.strip())
print(f"测试文件已写入: {test_file}")

In [ ]:
# 通过注册表调用自动发现的工具
result = await registry.invoke("run_pytest", {"path": str(test_file), "verbose": True})

print(f"success  : {result.success}")
print(f"exit_code: {result.metadata.get('exit_code')}")
print()
print("pytest 输出:")
print(result.content)

In [ ]:
# 清理演示用的测试文件
test_file.unlink(missing_ok=True)
print("测试文件已清理")

## 7. 多工具文件：再加一个工具

演示同时发现多个工具文件的情况。

In [ ]:
# 再写一个工具文件：系统信息工具
sysinfo_code = '''"""
自定义工具：获取系统信息。
"""
import asyncio
import platform
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent.parent.parent))

from src.tool_framework import Tool, ToolKind, ToolResult


class SysInfoTool(Tool):
    """获取当前系统的基本信息（OS、Python 版本、工作目录等）。"""

    @property
    def name(self) -> str:
        return "sys_info"

    @property
    def description(self) -> str:
        return "获取当前系统信息：操作系统、Python 版本、工作目录。用于调试环境问题。"

    @property
    def kind(self) -> ToolKind:
        return ToolKind.READ  # 只读，不修改任何状态

    def schema(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {},
                    "required": [],
                },
            },
        }

    def validate(self, params: dict) -> list[str]:
        return []  # 无参数，无需验证

    async def execute(self, params: dict) -> ToolResult:
        import os
        info = {
            "os": platform.system(),
            "os_version": platform.version(),
            "python": sys.version,
            "cwd": os.getcwd(),
            "platform": platform.platform(),
        }
        lines = [f"{k}: {v}" for k, v in info.items()]
        return ToolResult.ok("\\n".join(lines), **info)
'''

sysinfo_path = plugins_dir / "sys_info.py"
sysinfo_path.write_text(sysinfo_code.strip())
print(f"已写入: {sysinfo_path}")

# 列出插件目录里的所有文件
print("\n插件目录内容:")
for f in sorted(plugins_dir.glob("*.py")):
    print(f"  {f.name}")

In [ ]:
# 重新发现（模拟 Agent 重启）
registry2 = ToolRegistry()
manager2 = ToolDiscoveryManager(
    tools_dir=plugins_dir,
    registry=registry2,
)

registered = manager2.discover()

print(f"注册的工具: {registered}")
print(f"注册表: {registry2}")

In [ ]:
# 调用 sys_info 工具
result = await registry2.invoke("sys_info", {})
print(f"success: {result.success}")
print()
print(result.content)

## 8. 错误隔离：加载失败不崩溃

如果某个工具文件有 bug（语法错误、缺少依赖等），`ToolDiscoveryManager` 应该：
1. 捕获异常
2. 把错误信息记录到 `errors` 列表
3. 继续加载其他文件

不能因为一个坏文件让整个 Agent 崩溃。

In [ ]:
# 写一个有语法错误的工具文件
bad_tool_code = '''
# 这个文件有语法错误
from src.tool_framework import Tool, ToolKind, ToolResult

class BrokenTool(Tool
    # 缺少右括号，语法错误
    @property
    def name(self): return "broken"
'''

bad_path = plugins_dir / "broken_tool.py"
bad_path.write_text(bad_tool_code.strip())
print(f"已写入坏文件: {bad_path}")

# 写一个会在 import 时崩的文件
runtime_error_code = '''
# 这个文件 import 时会崩（缺少依赖）
import non_existent_package_xyz  # ImportError

from src.tool_framework import Tool, ToolKind, ToolResult

class AnotherTool(Tool):
    @property
    def name(self): return "another"
    @property
    def description(self): return "..."
    @property
    def kind(self): return ToolKind.READ
    def schema(self): return {}
    def validate(self, params): return []
    async def execute(self, params): return ToolResult.ok("ok")
'''

runtime_error_path = plugins_dir / "import_error_tool.py"
runtime_error_path.write_text(runtime_error_code.strip())
print(f"已写入会报 ImportError 的文件: {runtime_error_path}")

print("\n现在插件目录里有:")
for f in sorted(plugins_dir.glob("*.py")):
    print(f"  {f.name}")

In [ ]:
# 运行发现——坏文件不应该影响好文件
registry3 = ToolRegistry()
manager3 = ToolDiscoveryManager(tools_dir=plugins_dir, registry=registry3)

registered = manager3.discover()
print()
print("─" * 50)
print(f"成功注册的工具: {registered}")

errors = manager3.get_discovery_errors()
print(f"\n加载失败的文件: {len(errors)} 个")
for err in errors:
    print(f"  文件: {err['file'].name}")
    print(f"  错误: {err['error']}")
    print()

In [ ]:
# 确认好的工具仍然可以正常使用
print("好的工具仍然可用:")
for tool in registry3.list_tools():
    print(f"  {tool.name} ({tool.kind.value})")

In [ ]:
# 清理坏文件
bad_path.unlink(missing_ok=True)
runtime_error_path.unlink(missing_ok=True)
print("坏文件已清理")

## 9. 默认路径与工厂函数

实际使用时，Agent 启动时调用一行代码完成所有工具的发现和注册。

In [ ]:
def create_discovery_manager(
    registry: ToolRegistry,
    working_directory: str | Path = ".",
) -> ToolDiscoveryManager:
    """
    用默认路径创建 ToolDiscoveryManager。
    
    默认扫描目录: <working_directory>/.ai_agent/tools/
    
    Args:
        registry:          目标注册表
        working_directory: Agent 的工作目录，默认为当前目录
    Returns:
        配置好的 ToolDiscoveryManager
    """
    tools_dir = Path(working_directory) / ".ai_agent" / "tools"
    return ToolDiscoveryManager(tools_dir=tools_dir, registry=registry)


# 用法示例（Agent 启动代码里的样子）
reg = ToolRegistry()
dm = create_discovery_manager(reg, working_directory=".")

# 注册内置工具（正常流程）
# registry.register(ReadFileTool(cwd))
# registry.register(WriteFileTool(cwd))

# 自动发现自定义工具
names = dm.discover()
if dm.get_discovery_errors():
    print("警告：以下工具文件加载失败:")
    for err in dm.get_discovery_errors():
        print(f"  {err['file'].name}: {err['error']}")

print(f"\n自定义工具已加载: {names}")
print(f"当前注册表: {reg}")

## 10. 用 LLM 调用自动发现的工具

完整验证：LLM 能看到并调用通过插件机制加载的工具。

In [ ]:
import json
from src.llm_client import LLMClient
from src.context_manager import ContextManager

# 初始化
llm = LLMClient()
ctx = ContextManager(system_prompt="你是一个开发助手。有需要时使用工具获取信息。")

# 用自动发现的注册表
print(f"使用注册表: {reg}")
print(f"发给 LLM 的工具 schema 数量: {len(reg.get_schemas())}")
for schema in reg.get_schemas():
    print(f"  - {schema['function']['name']}")

In [ ]:
# 用 LLM 调用自动发现的 sys_info 工具
ctx.add_user_message("请用工具告诉我当前的系统信息（操作系统和 Python 版本）")

from openai import AsyncOpenAI
raw_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

response = await raw_client.chat.completions.create(
    model="gpt-oss:120b",
    messages=ctx.get_messages(),
    tools=reg.get_schemas(),
)

msg = response.choices[0].message
print(f"finish_reason: {response.choices[0].finish_reason}")
print(f"tool_calls: {msg.tool_calls}")

In [ ]:
if msg.tool_calls:
    # 把 LLM 的工具调用意图加回 context
    ctx.add_assistant_message(
        content=msg.content or "",
        tool_calls=[tc.model_dump() for tc in msg.tool_calls],
    )

    # 执行工具
    for tc in msg.tool_calls:
        name = tc.function.name
        params = json.loads(tc.function.arguments)
        print(f"LLM 调用工具: {name}，参数: {params}")

        result = await reg.invoke(name, params)
        print(f"工具结果 (success={result.success}):")
        print(result.content[:300])
        print()

        ctx.add_tool_result(tc.id, result.content)

    # 把工具结果发回 LLM，获取最终回答
    response2 = await raw_client.chat.completions.create(
        model="gpt-oss:120b",
        messages=ctx.get_messages(),
    )
    print("LLM 最终回答:")
    print(response2.choices[0].message.content)
else:
    print("LLM 直接回答（未调用工具）:")
    print(msg.content)

In [ ]:
await raw_client.close()
await llm.close()

## 11. 小结

| 模块 | 作用 |
|------|------|
| `load_module_from_file(path)` | 用 `importlib` 从任意 `.py` 文件动态加载模块 |
| `find_tools_in_module(module)` | 用 `dir()` 扫描模块，提取继承自 `Tool` 的类 |
| `ToolDiscoveryManager` | 扫描目录、逐文件加载、注册到 `ToolRegistry`，错误隔离 |
| `create_discovery_manager()` | 工厂函数，用默认路径 `.ai_agent/tools/` 创建管理器 |
| 错误隔离 | 每个文件独立 `try/except`，失败记录到 `errors`，不影响其他工具 |

**插件机制的好处**：框架代码和业务工具完全解耦。用户把 `.py` 文件放到约定目录，重启后自动可用。不需要修改框架代码，不需要了解注册流程。

**下一章**：Agent 记忆与会话持久化——把对话历史、工具调用记录保存到磁盘，让 Agent 重启后记得上次在做什么。

---
## 附：保存到 src/

In [ ]:
tool_discovery_code = '''
import importlib.util
import logging
import traceback
from pathlib import Path

from .tool_framework import Tool, ToolRegistry

logger = logging.getLogger(__name__)


def load_module_from_file(file_path: Path):
    """
    从指定 .py 文件动态加载模块。
    
    三步流程:
      1. spec_from_file_location: 创建模块规范（记录名称和位置）
      2. module_from_spec: 创建空模块对象
      3. exec_module: 执行代码，类/函数才真正存在于模块中
    """
    spec = importlib.util.spec_from_file_location(
        name=file_path.stem,
        location=str(file_path),
    )
    if spec is None or spec.loader is None:
        raise ImportError(f"无法为 {file_path} 创建模块规范")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def find_tools_in_module(module) -> list[type]:
    """
    扫描模块，返回所有继承自 Tool 的类（不含 Tool 基类本身）。
    """
    tools = []
    for name in dir(module):
        obj = getattr(module, name)
        if (
            isinstance(obj, type)
            and issubclass(obj, Tool)
            and obj is not Tool
        ):
            tools.append(obj)
    return tools


class ToolDiscoveryManager:
    """
    工具自动发现管理器。
    
    扫描 tools_dir 下的所有 .py 文件，找出继承自 Tool 的类，
    实例化后注册到 registry。一个文件加载失败不影响其他文件。
    """

    SKIP_FILES = {"__init__.py", "__main__.py"}

    def __init__(self, tools_dir: Path, registry: ToolRegistry):
        self.tools_dir = Path(tools_dir)
        self.registry = registry
        self._errors: list[dict] = []

    def discover(self) -> list[str]:
        """扫描目录，加载所有工具文件。返回成功注册的工具名列表。"""
        self._errors = []
        registered: list[str] = []

        if not self.tools_dir.exists():
            logger.info(f"工具目录不存在，跳过扫描: {self.tools_dir}")
            return registered

        py_files = sorted(self.tools_dir.glob("*.py"))

        for file_path in py_files:
            if file_path.name in self.SKIP_FILES:
                continue
            names = self._load_file(file_path)
            registered.extend(names)

        logger.info(
            f"发现完成：扫描 {len(py_files)} 个文件，"
            f"注册 {len(registered)} 个工具，"
            f"失败 {len(self._errors)} 个文件"
        )
        return registered

    def get_discovery_errors(self) -> list[dict]:
        """返回上一次 discover() 中加载失败的文件信息。"""
        return list(self._errors)

    def _load_file(self, file_path: Path) -> list[str]:
        """加载单个文件，失败时记录错误并返回空列表。"""
        try:
            module = load_module_from_file(file_path)
            tool_classes = find_tools_in_module(module)
            if not tool_classes:
                logger.debug(f"{file_path.name}: 未找到 Tool 子类，跳过")
                return []
            names = []
            for cls in tool_classes:
                instance = cls()
                self.registry.register(instance)
                names.append(instance.name)
                logger.info(f"已注册工具: {instance.name}  (来自 {file_path.name})")
            return names
        except Exception as e:
            tb = traceback.format_exc()
            self._errors.append({
                "file": file_path,
                "error": f"{type(e).__name__}: {e}",
                "traceback": tb,
            })
            logger.warning(f"加载失败: {file_path.name}  →  {type(e).__name__}: {e}")
            return []


def create_discovery_manager(
    registry: ToolRegistry,
    working_directory: str | Path = ".",
) -> ToolDiscoveryManager:
    """
    用默认路径创建 ToolDiscoveryManager。
    默认扫描目录: <working_directory>/.ai_agent/tools/
    """
    tools_dir = Path(working_directory) / ".ai_agent" / "tools"
    return ToolDiscoveryManager(tools_dir=tools_dir, registry=registry)
'''

import os
os.makedirs("../src", exist_ok=True)

with open("../src/tool_discovery.py", "w") as f:
    f.write(tool_discovery_code.strip())

print("已保存到 src/tool_discovery.py")
print()
print("后续用法:")
print("  from src.tool_discovery import ToolDiscoveryManager, create_discovery_manager")
print("  manager = create_discovery_manager(registry, working_directory='.')")
print("  manager.discover()")